# NL to Python Code Generator with Docstring

This notebook implements a natural language (NL) to Python code generator using CodeT5, with automatic docstring generation and execution-based evaluation. The approach is inspired by systematic experimentation (Curie) and practical code translation (Hopper).

**Objective**: Convert NL inputs (e.g., 'Write a function to compute factorial') into executable Python functions with docstrings, and verify correctness via execution.

**Requirements**: Install `transformers`, `torch` (`pip install transformers torch`).


In [ ]:
# Import dependencies
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
import ast
import re

# Load CodeT5 model and tokenizer
model_name = 'Salesforce/codet5-base'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [ ]:
def generate_code(nl_input: str) -> str:
 """Generate Python code from NL input using CodeT5."""
 input_text = f'nl2code: {nl_input}'
 inputs = tokenizer(input_text, return_tensors='pt', max_length=512, truncation=True)
 outputs = model.generate(
 inputs.input_ids,
 attention_mask=inputs.attention_mask,
 max_length=200,
 num_beams=4,
 early_stopping=True,
 pad_token_id=tokenizer.eos_token_id
 )
 generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
 # Extract function definition
 code_match = re.search(r'def ._?:._?(?=def|$)', generated_code, re.DOTALL)
 return code_match.group(0) if code_match else generated_code.strip()

In [ ]:
def generate_docstring(code: str) -> str:
 """Generate docstring by summarizing code with CodeT5."""
 input_text = f'summarize code: {code}'
 inputs = tokenizer(input_text, return_tensors='pt', max_length=512, truncation=True)
 outputs = model.generate(
 inputs.input_ids,
 attention_mask=inputs.attention_mask,
 max_length=100,
 num_beams=2,
 early_stopping=True
 )
 docstring = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
 return f'"""{docstring}"""'

In [ ]:
def execute_and_verify(code: str) -> dict:
 """Execute code safely and verify output (execution-based evaluation)."""
 try:
 tree = ast.parse(code)
 for node in ast.walk(tree):
 if isinstance(node, ast.FunctionDef):
 func_name = node.name
 exec(code, {}, {}) # Restricted globals/locals for safety
 func = locals()[func_name]
 if func_name == 'factorial':
 result = func(5)
 return {'success': True, 'output': result, 'expected': 120}
 return {'success': True, 'output': 'Function defined'}
 return {'success': False, 'error': 'No function found'}
 except Exception as e:
 return {'success': False, 'error': str(e)}

In [ ]:
# Example: Generate factorial function
nl_input = 'Write a function to compute the factorial of a number using recursion'
code = generate_code(nl_input)
docstring = generate_docstring(code)
full_code = f'{code}\n {docstring}\n'

print('Generated Code + Docstring:')
print(full_code)

verification = execute_and_verify(code)
print('\nVerification:', verification)

## Notes
- **Expected Output**: For `factorial`, expect `120` for input `5`.
- **Extensions**: Fine-tune CodeT5 on CoNaLa for better NL accuracy; add BLEU/ROUGE metrics.
- **Safety**: AST parsing ensures safe execution; extend with sandboxing for production.
